# Session 16: Building Custom Python Libraries

## Learning Objectives

By the end of this session, you will be able to:

1. Understand why and when to create a custom Python library
2. Structure a Python package with `__init__.py` and modules
3. Control the public API with `__init__.py` imports
4. Add documentation with docstrings
5. Manage dependencies with `pyproject.toml`
6. Install a package locally in editable mode

## Table of Contents

1. [Why Create a Library?](#1.-Why-Create-a-Library?)
2. [Packages vs Modules](#2.-Packages-vs-Modules)
3. [Your First Package: `mathkit`](#3.-Your-First-Package:-mathkit)
4. [The `__init__.py` File](#4.-The-__init__.py-File)
5. [Building a Real Library: `weathertools`](#5.-Building-a-Real-Library:-weathertools)
6. [Adding Documentation](#6.-Adding-Documentation)
7. [Managing Dependencies with `pyproject.toml`](#7.-Managing-Dependencies-with-pyproject.toml)
8. [Installing Locally with `pip install -e .`](#8.-Installing-Locally-with-pip-install--e-.)
9. [Brief Note: Publishing to PyPI](#9.-Brief-Note:-Publishing-to-PyPI)
10. [Exercises](#10.-Exercises)
11. [Summary](#11.-Summary)

---

## 1. Why Create a Library?

In the last session, you learned how to write standalone Python scripts. That's a huge step forward from notebooks — your code can now run from the terminal and accept arguments.

But what happens when you have **useful functions and classes** that you want to reuse across multiple scripts or projects? You have a few options:

| Approach | Pros | Cons |
|----------|------|------|
| **Copy-paste** | Quick, simple | Hard to maintain, inconsistencies |
| **Import from script** | Reusable, single source of truth | Must manage `sys.path`, messy for large projects |
| **Install as library** | Import from anywhere, versioned, shareable | Requires setup (but easy!) |

Think about the libraries you already use every day: `pandas`, `requests`, `scikit-learn`, `plotly`. These are all **packages that somebody built** — structured collections of Python code that you install and import.

In this session, you'll learn how to build your own.

---

## 2. Packages vs Modules

Before we start building, let's clarify the terminology.

### Modules

A **module** is simply a single `.py` file. You already know how to create and import these from Session 15:

```python
# my_module.py
def greet(name):
    return f"Hello, {name}!"
```

```python
# another_script.py
from my_module import greet
print(greet("Alice"))
```

### Packages

A **package** is a folder containing multiple modules plus a special `__init__.py` file:

```
mathkit/              <-- package (folder)
├── __init__.py       <-- makes it a package
├── stats.py          <-- module
└── geometry.py       <-- module
```

The `__init__.py` file is what tells Python "this folder is a package, not just a regular directory."

> **Note:** Think of it like this — a module is a single file, a package is a folder of related files. Like a toolbox with compartments.

### Summary of Terms

| Concept | What It Is | Example |
|---------|-----------|--------|
| **Module** | A single `.py` file | `stats.py` |
| **Package** | A folder with `__init__.py` | `mathkit/` |
| **Library** | An installable package | `pandas`, `requests` |

---

## 3. Your First Package: `mathkit`

We've prepared a simple package called `mathkit` in this session's folder. It contains two modules:

- `stats.py` — functions for `mean()`, `median()`, and `std_dev()`
- `geometry.py` — classes `Circle` and `Rectangle`

Let's explore it.

In [1]:
import os

for item in sorted(os.listdir("mathkit")):
    print(item)

__init__.py
__pycache__
geometry.py
stats.py


### Importing from Submodules

You can import directly from a specific module inside the package using dot notation:

In [2]:
from mathkit.stats import mean, median

data = [4, 8, 15, 16, 23, 42]
print(f"Mean: {mean(data)}")
print(f"Median: {median(data)}")

Mean: 18.0
Median: 15.5


In [3]:
from mathkit.geometry import Circle, Rectangle

c = Circle(5)
print(c)
print(f"Area: {c.area():.2f}")

r = Rectangle(4, 6)
print(r)
print(f"Perimeter: {r.perimeter()}")

Circle with radius 5 (area=78.54)
Area: 78.54
Rectangle (4 x 6, area=24.00)
Perimeter: 20


### Importing from the Package Directly

But what if we want to import the package itself and access everything through it? Can we do `import mathkit` and then use `mathkit.mean()`?

In [4]:
import mathkit

print(mathkit.mean([1, 2, 3]))
print(mathkit.Circle(3))

2.0
Circle with radius 3 (area=28.27)


This works because of `__init__.py` — let's see how.

---

## 4. The `__init__.py` File

The `__init__.py` file is the heart of a Python package. It runs automatically when the package is imported, and it serves two key purposes:

1. **Marks a folder as a package** — without it, Python won't recognize the folder as importable
2. **Defines the public API** — controls what users can import directly from the package

Let's look at the one inside `mathkit`:

In [5]:
# Let's look at what's inside mathkit/__init__.py
with open("mathkit/__init__.py") as f:
    print(f.read())

"""mathkit — A simple math utilities package."""

from .stats import mean, median, std_dev
from .geometry import Circle, Rectangle

__all__ = ["mean", "median", "std_dev", "Circle", "Rectangle"]



### Understanding Relative Imports

Notice the dot in `from .stats import mean` — the dot means **"from this package"**. This is called a **relative import**:

- `from .stats import mean` — import `mean` from the `stats` module **in this same package**
- `from .geometry import Circle` — import `Circle` from the `geometry` module **in this same package**

Without the dot, Python would look for a top-level module called `stats`, which is not what we want.

#### Relative vs Absolute Imports

There are two ways to import within a package: **relative** (with a dot) and **absolute** (with the full package name):

```python
# Relative import — uses a dot to mean "this package"
from .stats import mean

# Absolute import — uses the full package path
from mathkit.stats import mean
```

Both work, but **relative imports are preferred inside `__init__.py`** because they don't hardcode the package name. If you ever rename the package, relative imports keep working.

#### The Dot Notation Explained

The number of dots indicates how many levels **up** you're going in the package hierarchy:

| Syntax | Meaning | Example |
|--------|---------|---------|
| `from .module import x` | From **this** package | `from .stats import mean` |
| `from ..module import x` | From the **parent** package | Used in nested sub-packages |

For a flat package like `mathkit`, you'll only ever use a **single dot**. Double dots (`..`) become relevant when you have sub-packages (packages inside packages), which we won't cover in this course.

#### A Concrete Example

Imagine you're writing `mathkit/__init__.py`. Here's what Python sees when it encounters different import styles:

```python
# __init__.py inside mathkit/

from .stats import mean        # ✅ Looks for mathkit/stats.py → imports mean
from .geometry import Circle   # ✅ Looks for mathkit/geometry.py → imports Circle

from stats import mean         # ❌ Looks for a top-level stats.py (not inside mathkit!)
import stats                   # ❌ Same problem — ignores the package entirely
```

The dot is what anchors the import **relative to the current package's location**.

#### Where Can You Use Relative Imports?

Relative imports **only work inside packages** (folders with `__init__.py`). You **cannot** use them in standalone scripts:

```python
# my_script.py (standalone, not inside a package)
from .some_module import func   # ❌ ImportError: attempted relative import with no known parent package
```

This is another reason why packaging your code is useful — it unlocks relative imports, making your code more portable and self-contained.

#### One Module Importing from Another

Relative imports aren't just for `__init__.py`. Modules inside a package can import from each other too:

```python
# mathkit/geometry.py
from .stats import mean   # Import mean from the sibling module stats.py

class Circle:
    ...

    def some_method(self, values):
        return mean(values)  # Use a function from a sibling module
```

This keeps the package self-contained — modules collaborate with each other using relative imports.

### Understanding `__all__`

The `__all__` variable controls what gets exported when someone does `from mathkit import *`:

```python
__all__ = ["mean", "median", "std_dev", "Circle", "Rectangle"]
```

This is good practice — it makes the package's public API explicit and prevents internal implementation details from leaking out.

### How `__init__.py` Content Affects Imports

| `__init__.py` Content | Effect |
|----------------------|--------|
| Empty file | `import mathkit` works but exposes nothing directly |
| `from .stats import mean` | `mathkit.mean()` and `from mathkit import mean` work |
| `__all__ = ["mean"]` | `from mathkit import *` only exports `mean` |

In [6]:
# Because of __init__.py, all of these work:
from mathkit import mean, median, std_dev, Circle, Rectangle

print(f"Mean of [10, 20, 30]: {mean([10, 20, 30])}")
print(f"Std dev of [10, 20, 30]: {std_dev([10, 20, 30]):.2f}")
print(f"Circle(7) area: {Circle(7).area():.2f}")
print(f"Rectangle(3, 5) is square? {Rectangle(3, 5).is_square}")

Mean of [10, 20, 30]: 20.0
Std dev of [10, 20, 30]: 8.16
Circle(7) area: 153.94
Rectangle(3, 5) is square? False


---

## 5. Building a Real Library: `weathertools`

The `mathkit` package was a simple "flat" package — just a folder with modules. For a real, installable library, we use the **src-layout**, which is the modern Python packaging standard.

Here's the structure of our `weathertools` library:

```
weather-tools/                 <-- project directory
├── pyproject.toml             <-- project metadata & dependencies
├── README.md                  <-- documentation
└── src/
    └── weathertools/          <-- the actual package
        ├── __init__.py
        ├── converters.py
        └── forecast.py
```

### Why the `src/` Layout?

The `src/` layout prevents a subtle but common bug: without it, Python might accidentally import from the local folder instead of the installed package. By putting the code inside `src/`, you guarantee that imports only work after the package is properly installed.

> **Note:** Notice that the project directory (`weather-tools/`) has a different name from the package (`weathertools/`). This is a common convention — the project folder can be named anything, while the package name (inside `src/`) is what you use in `import` statements. Using different names avoids conflicts where Python might confuse the project folder for the package itself.

Let's look at the code inside this library.

In [ ]:
# converters.py — temperature conversion functions
with open("weather-tools/src/weathertools/converters.py") as f:
    print(f.read())

In [ ]:
# forecast.py — the WeatherForecast class
with open("weather-tools/src/weathertools/forecast.py") as f:
    print(f.read())

In [ ]:
# __init__.py — the public API
with open("weather-tools/src/weathertools/__init__.py") as f:
    print(f.read())

Notice the same pattern as `mathkit`: the `__init__.py` imports from submodules and defines `__all__`.

> **Note:** We can't import `weathertools` yet by just doing `import weathertools`. Because it uses the `src/` layout, Python doesn't know where to find it. We need to **install** it first — we'll do that in Section 8.

---

## 6. Adding Documentation

Good documentation makes your library usable — not just by others, but by your future self. Python has a built-in documentation system: **docstrings**.

### Three Levels of Docstrings

1. **Module docstring** — at the top of a `.py` file, describes the module
2. **Class docstring** — right after `class ClassName:`, describes the class
3. **Function docstring** — right after `def function_name():`, describes the function

### Google-Style Docstrings

There are several docstring formats. We'll use the **Google style**, which is clean and readable:

```python
def celsius_to_fahrenheit(celsius):
    """Convert a temperature from Celsius to Fahrenheit.

    Args:
        celsius: Temperature in degrees Celsius.

    Returns:
        Temperature in degrees Fahrenheit.
    """
    return celsius * 9 / 5 + 32
```

The key sections are:
- **Summary line** — first line, brief description of what the function does
- **Args** — describes each parameter
- **Returns** — describes the return value
- **Raises** — (optional) describes exceptions that may be raised

### Viewing Documentation with `help()`

Python's built-in `help()` function reads docstrings and displays them in a formatted way:

In [ ]:
from mathkit.stats import mean

help(mean)

Help on function mean in module mathkit.stats:

mean(numbers)
    Calculate the arithmetic mean of a list of numbers.



In [ ]:
from mathkit.geometry import Circle

help(Circle)

Help on class Circle in module mathkit.geometry:

class Circle(builtins.object)
 |  Circle(radius)
 |  
 |  A circle defined by its radius.
 |  
 |  Methods defined here:
 |  
 |  __init__(self, radius)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self)
 |      Return repr(self).
 |  
 |  __str__(self)
 |      Return str(self).
 |  
 |  area(self)
 |      Calculate the area of the circle.
 |  
 |  perimeter(self)
 |      Calculate the perimeter (circumference) of the circle.
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables
 |  
 |  __weakref__
 |      list of weak references to the object



### Type Hints as Additional Documentation

Type hints tell the reader (and their IDE) what types a function expects and returns:

```python
def mean(numbers: list[float]) -> float:
    """Calculate the arithmetic mean."""
    ...
```

> **Note:** Type hints are optional but helpful. They don't change how the code runs — they're documentation for humans and IDEs.

---

## 7. Managing Dependencies with `pyproject.toml`

When you build a library, you need a way to declare:
- What your project is called
- What version it is
- What other packages it depends on
- How to build and install it

The modern standard for all of this is **`pyproject.toml`**.

Let's look at the one in our `weathertools` project:

In [ ]:
with open("weather-tools/pyproject.toml") as f:
    print(f.read())

### Understanding Each Section

- **`[build-system]`** — tells `pip` how to build the package. `setuptools` is the most common build backend.
- **`[project]`** — metadata about the project:
  - `name` — the package name (what you use in `pip install <name>`)
  - `version` — the version number (follows [semantic versioning](https://semver.org/))
  - `description` — a short description
  - `requires-python` — minimum Python version
  - `dependencies` — other packages your library needs (empty for `weathertools` since it only uses the standard library)

### Historical Context

You might see older projects using different files for configuration. Here's how they compare:

| File | Purpose | Modern? |
|------|---------|--------|
| `requirements.txt` | Lists dependencies | Older approach |
| `setup.py` | Build configuration | Legacy |
| `pyproject.toml` | All-in-one configuration | Current standard |

> **Note:** `pyproject.toml` was introduced in [PEP 518](https://peps.python.org/pep-0518/) and is now the recommended way to configure Python projects. You'll still see `requirements.txt` and `setup.py` in older projects.

---

## 8. Installing Locally with `pip install -e .`

Now comes the magic. We can **install** our `weathertools` library locally so that we can import it from anywhere — just like `pandas` or `requests`.

The key command is:

```bash
pip install -e .
```

The `-e` flag stands for **editable mode**. This means:
- The package is installed by creating a link to your source code
- Any changes you make to the source files take effect **immediately** — no need to reinstall
- Perfect for development!

> **Note:** Since our environment is managed by `uv`, we use `uv pip install` instead of plain `pip install`. This ensures the package is installed into the correct virtual environment. The `-e` flag works exactly the same way.

Let's install `weathertools`:

In [ ]:
!uv pip install -e ./weather-tools

Now we can import and use it just like any other library:

In [14]:
from weathertools import celsius_to_fahrenheit, WeatherForecast

# Convert temperature
print(f"25°C = {celsius_to_fahrenheit(25)}°F")

# Create a forecast
forecast = WeatherForecast("Madrid")
forecast.add_day("2026-02-23", high=12, low=4, condition="Cloudy")
forecast.add_day("2026-02-24", high=15, low=6, condition="Sunny")
forecast.add_day("2026-02-25", high=18, low=8, condition="Sunny")
forecast.add_day("2026-02-26", high=10, low=3, condition="Rainy")
print()
print(forecast.summary())

ImportError: cannot import name 'celsius_to_fahrenheit' from 'weathertools' (unknown location)

Let's verify it's properly installed:

In [ ]:
!uv pip show weathertools

---

## 9. Brief Note: Publishing to PyPI

So far, we've installed our library **locally**. But what if you want to share it with the world?

**PyPI** (Python Package Index) is the public repository at [pypi.org](https://pypi.org) where all the packages you `pip install` come from. Publishing to PyPI makes your library available to anyone via `pip install your-library-name`.

The high-level process is:

1. **Build** the distribution files: `python -m build`
2. **Upload** to PyPI: `twine upload dist/*` or `uv publish`
3. Anyone can now `pip install your-library-name`

We won't cover this hands-on — it requires creating a PyPI account and is beyond our scope. But now you know it exists and how the ecosystem works.

> For more details, see the [Python Packaging Guide](https://packaging.python.org/en/latest/tutorials/packaging-projects/).

---

## 10. Exercises

### Exercise 1: `textutils` Package

Create a `textutils` package with two modules:

1. **`cleaning.py`** with the following functions:
   - `remove_punctuation(text)` — remove all punctuation from a string
   - `normalize_whitespace(text)` — replace multiple spaces/tabs/newlines with a single space and strip leading/trailing whitespace
   - `to_snake_case(text)` — convert a string like `"Hello World"` to `"hello_world"`

2. **`analysis.py`** with the following functions:
   - `word_count(text)` — return the number of words
   - `char_count(text)` — return the number of characters (excluding spaces)
   - `most_common_word(text)` — return the most frequently occurring word (lowercase)

3. **`__init__.py`** that exports all six functions

Your folder structure should be:
```
textutils/
├── __init__.py
├── cleaning.py
└── analysis.py
```

**Hints:**
- For `remove_punctuation`, you can use `string.punctuation` from the `string` module
- For `normalize_whitespace`, look at `str.split()` and `" ".join()`
- For `most_common_word`, consider using `collections.Counter`

In [ ]:
# Create the textutils folder
import os

os.makedirs("textutils", exist_ok=True)

In [ ]:
%%writefile textutils/cleaning.py
"""Text cleaning functions for textutils."""


def remove_punctuation(text):
    """Remove all punctuation from a string."""
    pass


def normalize_whitespace(text):
    """Replace multiple spaces/tabs/newlines with a single space."""
    pass


def to_snake_case(text):
    """Convert a string to snake_case."""
    pass

In [ ]:
%%writefile textutils/analysis.py
"""Text analysis functions for textutils."""


def word_count(text):
    """Return the number of words in the text."""
    pass


def char_count(text):
    """Return the number of characters, excluding spaces."""
    pass


def most_common_word(text):
    """Return the most frequently occurring word (lowercase)."""
    pass

In [ ]:
%%writefile textutils/__init__.py
"""textutils — A text processing utilities package."""

# TODO: Import all functions from .cleaning and .analysis
# TODO: Define __all__

pass

In [ ]:
# After implementing, test your package:
# from textutils import remove_punctuation, word_count, most_common_word
#
# print(remove_punctuation("Hello, World!"))           # "Hello World"
# print(word_count("the cat sat on the mat"))           # 6
# print(most_common_word("the cat sat on the mat"))     # "the"

### Exercise 2: Add Docstrings

Go back to your `textutils` package and add **Google-style docstrings** to all six functions. Each docstring should include:

1. A summary line
2. An `Args` section describing each parameter
3. A `Returns` section describing the return value

Example for reference:

```python
def remove_punctuation(text):
    """Remove all punctuation characters from a string.

    Args:
        text: The input string to clean.

    Returns:
        A new string with all punctuation removed.
    """
```

After adding docstrings, verify they work with `help()`:

In [ ]:
# After adding docstrings, verify:
# from textutils.cleaning import remove_punctuation
# help(remove_punctuation)

### Exercise 3 (Homework): `financekit` Library

Create a full, installable `financekit` library using the `src/` layout. It should include:

1. **`models.py`** module with:
   - `Transaction` class — attributes: `date` (str), `amount` (float), `category` (str)
   - `Portfolio` class — attributes: `name` (str), `transactions` (list); methods: `add_transaction()`, `total_value()`, `filter_by_category(category)`

2. **`analysis.py`** module with:
   - `monthly_summary(transactions)` — returns a dict mapping month strings to total amounts
   - `category_breakdown(transactions)` — returns a dict mapping categories to total amounts

3. **`__init__.py`** — exports key classes and functions

4. **`pyproject.toml`** — proper project metadata

5. Install locally in editable mode and verify

Your folder structure should be:

```
financekit/
├── pyproject.toml
└── src/
    └── financekit/
        ├── __init__.py
        ├── models.py
        └── analysis.py
```

Here is skeleton code for each file:

**`pyproject.toml`:**

```toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "financekit"
version = "0.1.0"
description = "A simple finance utilities library."
requires-python = ">=3.11"
dependencies = []
```

**`models.py`:**

```python
"""Financial models for financekit."""


class Transaction:
    """Represents a single financial transaction.

    Args:
        date: Date string in 'YYYY-MM-DD' format.
        amount: Transaction amount (positive for income, negative for expense).
        category: Category of the transaction (e.g., 'Food', 'Salary').
    """

    def __init__(self, date, amount, category):
        pass

    def __repr__(self):
        pass


class Portfolio:
    """A collection of financial transactions.

    Args:
        name: Name of the portfolio.
    """

    def __init__(self, name):
        pass

    def add_transaction(self, transaction):
        """Add a transaction to the portfolio."""
        pass

    def total_value(self):
        """Calculate the total value of all transactions."""
        pass

    def filter_by_category(self, category):
        """Return transactions matching the given category."""
        pass
```

**`analysis.py`:**

```python
"""Financial analysis functions for financekit."""


def monthly_summary(transactions):
    """Summarize transactions by month.

    Args:
        transactions: List of Transaction objects.

    Returns:
        Dict mapping month strings ('YYYY-MM') to total amounts.
    """
    pass


def category_breakdown(transactions):
    """Summarize transactions by category.

    Args:
        transactions: List of Transaction objects.

    Returns:
        Dict mapping category names to total amounts.
    """
    pass
```

**`__init__.py`:**

```python
"""financekit — A simple finance utilities library."""

from .models import Transaction, Portfolio
from .analysis import monthly_summary, category_breakdown

__all__ = ["Transaction", "Portfolio", "monthly_summary", "category_breakdown"]
```

**Test your library after installing:**

```python
from financekit import Transaction, Portfolio, category_breakdown

p = Portfolio("My Budget")
p.add_transaction(Transaction("2026-01-15", -50.0, "Food"))
p.add_transaction(Transaction("2026-01-20", 2000.0, "Salary"))
p.add_transaction(Transaction("2026-01-22", -30.0, "Food"))
p.add_transaction(Transaction("2026-02-01", 2000.0, "Salary"))

print(f"Total: {p.total_value()}")            # 3920.0
print(f"Food: {p.filter_by_category('Food')}") # [Transaction(...), Transaction(...)]
print(category_breakdown(p.transactions))       # {'Food': -80.0, 'Salary': 4000.0}
```

---

## 11. Summary

In this session, you learned:

1. **Why create libraries**: Reuse code, share with others, avoid copy-pasting
2. **Packages vs modules**: A package is a folder with `__init__.py` containing multiple modules
3. **`__init__.py`**: Controls the public API of your package
4. **Documentation**: Google-style docstrings + `help()` to view them
5. **`pyproject.toml`**: Modern standard for project configuration and dependencies
6. **Editable install**: `pip install -e .` for development workflow

### Key Takeaways

- A package is just a folder with an `__init__.py` file — nothing magical
- Use `__init__.py` to define what users can import directly from your package
- The `src/` layout is the modern standard for installable packages
- `pip install -e .` lets you develop and test without reinstalling
- Good docstrings make your library self-documenting

### What's Next?

You now have the skills to:
- Write standalone Python scripts (S15)
- Package your code into reusable libraries (S16)
- Build data applications with Streamlit (S14)

These are the building blocks for professional Python development!